In [0]:
dbutils.widgets.dropdown("use_unity_catalog", "true", ["true", "false"], "Use Unity Catalog")
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_prefix", "retail", "Schema Prefix")
dbutils.widgets.text("landing_path", "/Volumes/workspace/default/retail_landing", "Landing Path")

In [0]:
catalog = dbutils.widgets.get("catalog_name")
schema_prefix = dbutils.widgets.get("schema_prefix")
landing_path = dbutils.widgets.get("landing_path")
bronze_db = f"{catalog}.{schema_prefix}_bronze"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {bronze_db}")
print("Bronze schema:", bronze_db, "| Landing path:", landing_path)

In [0]:
%run ./00b_pipeline_utils

In [0]:
datasets = [
    {"prefix": "olist_customers_dataset", "table": "raw_customers"},
    {"prefix": "olist_geolocation_dataset", "table": "raw_geolocation"},
    {"prefix": "olist_order_items_dataset", "table": "raw_order_items"},
    {"prefix": "olist_order_payments_dataset", "table": "raw_order_payments"},
    {"prefix": "olist_order_reviews_dataset", "table": "raw_order_reviews"},
    {"prefix": "olist_orders_dataset", "table": "raw_orders"},
    {"prefix": "olist_products_dataset", "table": "raw_products"},
    {"prefix": "olist_sellers_dataset", "table": "raw_sellers"},
    {"prefix": "product_category_name_translation", "table": "raw_product_category_name_translation"},
]

In [0]:
from pyspark.sql.functions import current_timestamp, col

def ingest_to_bronze(prefix, target_table):
    schema_loc = f"{landing_path}/_checkpoints/{target_table}/schema"
    checkpoint_loc = f"{landing_path}/_checkpoints/{target_table}/checkpoint"

    stream_df = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "csv")
                 .option("cloudFiles.schemaLocation", schema_loc)
                 .option("header", "true")
                 .option("multiLine", "true")
                 .option("quote", '"')
                 .option("escape", '"')
                 .option("cloudFiles.inferColumnTypes", "true")
                 .option("pathGlobFilter", f"{prefix}*.csv")
                 .load(landing_path))

    processed_df = (stream_df
                    .withColumn("_ingestion_timestamp", current_timestamp())
                    .withColumn("_source_file_name", col("_metadata.file_path")))

    query = (processed_df.writeStream
             .format("delta")
             .option("checkpointLocation", checkpoint_loc)
             .option("mergeSchema", "true")
             .outputMode("append")
             .trigger(availableNow=True)
             .toTable(f"{bronze_db}.{target_table}"))

    query.awaitTermination()
    return spark.table(f"{bronze_db}.{target_table}").count()

In [0]:
succeeded, failed = [], []

for d in datasets:
    log_event("bronze", d["table"], "START", f"Auto Loader scanning for {d['prefix']}*.csv")
    try:
        row_count = ingest_to_bronze(d["prefix"], d["table"])
        log_event("bronze", d["table"], "SUCCESS", rows_affected=row_count)
        succeeded.append(d["table"])
    except Exception as e:
        log_event("bronze", d["table"], "FAIL", message=str(e))
        failed.append(d["table"])

print(f"\nSucceeded: {succeeded}")
print(f"Failed: {failed if failed else 'none'}")
if len(failed) == len(datasets):
    raise Exception("Bronze ingestion failed for ALL datasets.")

In [0]:
display(spark.sql(f"SHOW TABLES IN {bronze_db}"))